In [1]:
run_origen = ""

In [2]:
# Parameters
run_origen = "scheduled__2026-09-28T01:07:14.573066+00:00"


In [3]:
import json
import uuid

import numpy as np
import psycopg2
import sklearn

from psycopg2.extras import Json
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
)

from cobertura_common import (
    ENTRADAS,
    prefijo,
    leer_objeto,
    guardar_objeto,
    guardar_bytes,
    asegurar_bucket,
)

if not run_origen:
    raise ValueError(
        "Airflow debe enviar el run_id mediante run_origen."
    )

db = psycopg2.connect()

try:
    with db.cursor() as cursor:
        cursor.execute("""
            SELECT features, etiqueta, conjunto
            FROM cobertura_ready
            WHERE run_id = %s
            ORDER BY row_id
        """, (run_origen,))

        filas = cursor.fetchall()

        cursor.execute("""
            SELECT count(DISTINCT batch)
            FROM cobertura_ingestas
            WHERE grupo = 3
        """)

        batches = cursor.fetchone()[0]
finally:
    db.close()

if not filas:
    raise ValueError(
        f"No hay datos preparados para {run_origen}"
    )

X = np.asarray([fila[0] for fila in filas], dtype=float)
y = np.asarray([fila[1] for fila in filas], dtype=int)
train = np.asarray([fila[2] == "train" for fila in filas])
test = ~train

if not train.any() or not test.any():
    raise ValueError("Faltan datos de entrenamiento o prueba.")

# Recupera el preprocesador de ESTA ejecución.
preprocesador = leer_objeto(
    prefijo(run_origen) + "/preprocesador.joblib"
)

# Mantiene compatible la celda de métricas que ya tienes.
paquete_anterior = {
    "metricas": {
        "batches_recolectados": int(batches)
    }
}

print("Ejecución:", run_origen)
print("Entrenamiento:", train.sum())
print("Prueba:", test.sum())

Ejecución: scheduled__2026-09-28T01:07:14.573066+00:00
Entrenamiento: 2456
Prueba: 602


In [4]:
modelo = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=2,
)

modelo.fit(X[train], y[train])
predicciones = modelo.predict(X[test])

metricas = {
    "accuracy": float(accuracy_score(y[test], predicciones)),
    "f1_macro": float(f1_score(
        y[test],
        predicciones,
        average="macro",
        zero_division=0,
    )),
    "filas_train": int(train.sum()),
    "filas_test": int(test.sum()),
    "batches_recolectados": (
        paquete_anterior["metricas"]["batches_recolectados"]
    ),
    "clases_entrenadas": modelo.classes_.tolist(),
    "clases_test": np.unique(y[test]).tolist(),
    "matriz_confusion_orden_1_a_7": confusion_matrix(
        y[test], predicciones, labels=list(range(1, 8))
    ).tolist(),
}

print(json.dumps(metricas, indent=2))

{
  "accuracy": 0.9634551495016611,
  "f1_macro": 0.8431038961038961,
  "filas_train": 2456,
  "filas_test": 602,
  "batches_recolectados": 1,
  "clases_entrenadas": [
    1,
    2,
    4,
    5,
    6
  ],
  "clases_test": [
    1,
    2,
    4,
    5,
    6
  ],
  "matriz_confusion_orden_1_a_7": [
    [
      514,
      0,
      0,
      4,
      1,
      2,
      0
    ],
    [
      4,
      20,
      0,
      0,
      0,
      0,
      0
    ],
    [
      0,
      0,
      0,
      0,
      0,
      0,
      0
    ],
    [
      2,
      0,
      0,
      21,
      0,
      0,
      0
    ],
    [
      9,
      0,
      0,
      0,
      5,
      0,
      0
    ],
    [
      0,
      0,
      0,
      0,
      0,
      20,
      0
    ],
    [
      0,
      0,
      0,
      0,
      0,
      0,
      0
    ]
  ]
}


In [5]:
asegurar_bucket()

version = "jupyter_" + uuid.uuid4().hex
carpeta = f"grupo3/{version}"
objeto = f"{carpeta}/modelo.joblib"

paquete = {
    "modelo": modelo,
    "preprocesador": preprocesador,
    "columnas": ENTRADAS,
    "metricas": metricas,
    "run_id": version,
    "run_id_datos": run_origen,
    "sklearn_version": sklearn.__version__,
}

checksum = guardar_objeto(objeto, paquete)

guardar_bytes(
    f"{carpeta}/metricas.json",
    json.dumps(metricas).encode(),
    "application/json",
)

db = psycopg2.connect()
try:
    with db:
        with db.cursor() as cursor:
            cursor.execute("""
                INSERT INTO cobertura_modelos
                    (run_id, objeto, metricas)
                VALUES (%s, %s, %s)
            """, (version, objeto, Json(metricas)))
finally:
    db.close()

print("Modelo guardado:", objeto)

Modelo guardado: grupo3/jupyter_f8f2eb16c8ff451d9c5009cabe146fc8/modelo.joblib


In [6]:
nuevo_manifiesto = {
    "objeto": objeto,
    "sha256": checksum,
    "run_id": version,
    "metricas": metricas,
}

guardar_bytes(
    "grupo3/latest.json",
    json.dumps(nuevo_manifiesto).encode(),
    "application/json",
)

print("Modelo activado:", version)

Modelo activado: jupyter_f8f2eb16c8ff451d9c5009cabe146fc8
